In [1]:
import cv2
import numpy as np

# ==========================================================
# Sky & Cloud Parameters (LAB Space)
# ==========================================================
NIGHT_THRESHOLD = 60
DARK_L_THRESHOLD = 100
WHITE_L_THRESHOLD = 200
PIXEL_COUNT_THRESHOLD = 200
SUN_SEARCH_RADIUS = 50


def analyze_sky_and_clouds(frame):
    """
    تحلیل وضعیت آسمان و ابر بر اساس فضای رنگی LAB
    خروجی‌ها: (is_day, details)
    """
    if frame is None or frame.size == 0:
        return None, {"time": "Error", "cloud_status": "Invalid Frame"}

    h, w = frame.shape[:2]

    # ۱. بررسی شب بر اساس ۲۰ درصد بالای تصویر (کانال V در HSV)
    sky_top_roi = frame[:int(0.2 * h), :]
    hsv_top = cv2.cvtColor(sky_top_roi, cv2.COLOR_BGR2HSV)
    mean_val_top = np.mean(hsv_top[:, :, 2])

    if mean_val_top < NIGHT_THRESHOLD:
        return False, {
            "time": "Night",
            "cloud_status": "N/A",
            "dark_count": 0,
            "white_count": 0,
            "sun_center": None
        }

    # ۲. تحلیل شرایط روز بر اساس کانال Lightness در فضای LAB
    lab = cv2.cvtColor(frame, cv2.COLOR_BGR2LAB)
    l_channel = lab[:, :, 0]

    # یافتن درخشان‌ترین نقطه (تخمین خورشید)
    blurred_l = cv2.GaussianBlur(l_channel, (11, 11), 0)
    _, _, _, sun_center = cv2.minMaxLoc(blurred_l)

    # ایجاد ماسک دایره‌ای اطراف خورشید
    mask = np.zeros_like(l_channel, dtype=np.uint8)
    cv2.circle(mask, sun_center, SUN_SEARCH_RADIUS, 255, -1)
    mask_bool = mask > 0

    # شمارش پیکسل‌های تیره و روشن در محدوده
    dark_count = int(np.sum((l_channel <= DARK_L_THRESHOLD) & mask_bool))
    white_count = int(np.sum((l_channel >= WHITE_L_THRESHOLD) & mask_bool))

    # ۳. تفکیک صرفاً بین ابر سیاه و ابر سفید
    if dark_count >= PIXEL_COUNT_THRESHOLD:
        cloud_status = "Dark Clouds"
    else:
        cloud_status = "White Clouds"

    details = {
        "time": "Day",
        "cloud_status": cloud_status,
        "dark_count": dark_count,
        "white_count": white_count,
        "sun_center": sun_center
    }

    return True, details

# ==========================================================
# Detection Parameters
# ==========================================================
MIN_SUN_AREA_RATIO = 0.0002
MAX_SUN_AREA_RATIO = 0.6

MIN_CIRCULARITY = 0.5
MIN_SOLIDITY = 0.2

MIN_ASPECT_RATIO = 0.60
MAX_ASPECT_RATIO = 1.4

LOCAL_CONTRAST_LIMIT = 2
SUN_BRIGHTNESS_RATIO = 0.98
MIN_HOUGH_SCORE = 150.0  # آستانه حداقل امتیاز برای پذیرش دایره هاف


def detect_sun_white(frame, offset_x=0, offset_y=0):
    if frame is None or frame.size == 0:
        return None, None, None, None, None

    # ۱. بررسی اولیه وضعیت شب/روز و نوع ابر
    is_day, details = analyze_sky_and_clouds(frame)
    
    # اگر شب بود هیچ پردازشی انجام نده
    if not is_day:
        return None, None, None, None, None

    h, w = frame.shape[:2]

    # محدود کردن ناحیه پردازش به ۹۰٪ بالای تصویر در حالت کلی
    if offset_x == 0 and offset_y == 0:
        sky_limit = int(h * 0.90)
        frame = frame[:sky_limit]
        h, w = frame.shape[:2]

    total_pixels = h * w

    # ==========================================================
    # پردازش مخصوص حالت: ابر سفید / روز (White Clouds)
    # ==========================================================
    if details["cloud_status"] == "White Clouds":
        # تبدیل به فضای رنگی LAB و استفاده مستقیم از کانال L
        lab = cv2.cvtColor(frame, cv2.COLOR_BGR2LAB)
        l_channel = lab[:, :, 0]
        
        blur_l = cv2.GaussianBlur(l_channel, (9, 9), 0)

        # آستانه‌گذاری پویا روی کانال L
        max_val = np.max(blur_l)
        custom_thresh_val = int(max_val * SUN_BRIGHTNESS_RATIO)
        _, mask_l = cv2.threshold(l_channel, custom_thresh_val, 255, cv2.THRESH_BINARY)

        # عملیات مورفولوژی
        kernel1 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
        kernel2 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (13, 13))

        mask = cv2.morphologyEx(mask_l, cv2.MORPH_OPEN, kernel1)
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel2)

        contours, _ = cv2.findContours(
            mask,
            cv2.RETR_EXTERNAL,
            cv2.CHAIN_APPROX_SIMPLE
        )

        best = None
        best_score = -1
        best_contour = None
        
        # پرچم برای بررسی وجود کانتور معتبر اما غیر گرد (به دلیل پیوستگی به ابر سفید)
        has_deformed_candidate = False

        # -------------------------------------------------
        # ارزیابی کانتورها
        # -------------------------------------------------
        for cnt in contours:
            area = cv2.contourArea(cnt)
            if area < total_pixels * MIN_SUN_AREA_RATIO or area > total_pixels * MAX_SUN_AREA_RATIO:
                continue

            perimeter = cv2.arcLength(cnt, True)
            if perimeter <= 0:
                continue

            circularity = (4.0 * np.pi * area) / (perimeter * perimeter)

            hull = cv2.convexHull(cnt)
            hull_area = cv2.contourArea(hull)
            if hull_area <= 0:
                continue

            solidity = area / hull_area
            if solidity < MIN_SOLIDITY:
                continue

            x, y, bw, bh = cv2.boundingRect(cnt)
            if bh == 0:
                continue

            aspect_ratio = bw / float(bh)
            if aspect_ratio < MIN_ASPECT_RATIO or aspect_ratio > MAX_ASPECT_RATIO:
                continue

            M = cv2.moments(cnt)
            if M["m00"] == 0:
                continue

            cx = int(M["m10"] / M["m00"])
            cy = int(M["m01"] / M["m00"])

            object_mask = np.zeros_like(mask)
            cv2.drawContours(object_mask, [cnt], -1, 255, -1)

            mean_l_value = cv2.mean(l_channel, mask=object_mask)[0]

            # محاسبه کنتراست محلی روی کانال L
            ring_mask = cv2.dilate(
                object_mask,
                cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (31, 31))
            )
            ring_mask = cv2.subtract(ring_mask, object_mask)
            background_l_value = cv2.mean(l_channel, mask=ring_mask)[0]
            local_contrast = mean_l_value - background_l_value

            if local_contrast < LOCAL_CONTRAST_LIMIT:
                continue

            # اگر تمام شرایط برقرار بود اما گردی کانتور کم بود، ثبت می‌کنیم که کانتور تغییرشکل‌یافته داریم
            if circularity < MIN_CIRCULARITY:
                has_deformed_candidate = True
                continue

            mean_bgr = cv2.mean(frame, mask=object_mask)[:3]
            (_, radius) = cv2.minEnclosingCircle(cnt)
            radius = int(radius)

            score = (
                mean_l_value * 1.8 +
                local_contrast * 8.0 +
                circularity * 250 +
                solidity * 250 +
                np.sqrt(area)
            )

            if score > best_score:
                best_score = score
                best_contour = cnt
                best = {
                    "center": (cx + offset_x, cy + offset_y),
                    "radius_est": radius,
                    "pixel_count": int(area),
                    "mean_bgr": mean_bgr,
                    "mean_value": mean_l_value,
                    "local_contrast": local_contrast,
                    "circularity": circularity,
                    "solidity": solidity,
                    "score": score,
                    "method": "contour"
                }

        # -------------------------------------------------
        # Fallback: Hough Circles (فقط در صورت وجود کانتور نامنظم/غیر گرد)
        # -------------------------------------------------
        if best is None and has_deformed_candidate:
            hough_input = cv2.GaussianBlur(mask, (5, 5), 0)
            
            min_r = int(np.sqrt(total_pixels * MIN_SUN_AREA_RATIO / np.pi))
            max_r = int(np.sqrt(total_pixels * MAX_SUN_AREA_RATIO / np.pi))
            
            circles = cv2.HoughCircles(
                hough_input,
                cv2.HOUGH_GRADIENT,
                dp=1.2,
                minDist=30,
                param1=80,
                param2=10,
                minRadius=max(5, min_r),
                maxRadius=max_r
            )

            if circles is not None:
                circles = np.uint16(np.around(circles))
                best_hough = None
                best_hough_score = -1.0

                for c in circles[0, :]:
                    cx_h, cy_h, r_h = int(c[0]), int(c[1]), int(c[2])
                    
                    if 0 <= cx_h < w and 0 <= cy_h < h and r_h > 0:
                        temp_circle_mask = np.zeros_like(mask)
                        cv2.circle(temp_circle_mask, (cx_h, cy_h), r_h, 255, -1)
                        
                        overlap = cv2.bitwise_and(mask, temp_circle_mask)
                        bright_pixels = cv2.countNonZero(overlap)
                        
                        if bright_pixels == 0:
                            continue

                        circle_area = np.pi * (r_h ** 2)
                        density = bright_pixels / circle_area
                        mean_l_inside = cv2.mean(l_channel, mask=temp_circle_mask)[0]

                        # محاسبه امتیاز ترکیبی برای هاف
                        hough_score = (bright_pixels * density * 0.5) + (mean_l_inside * 1.2)

                        # اعمال شرط ارزیابی و امتیاز حداقل برای پذیرش هاف
                        if hough_score > MIN_HOUGH_SCORE and hough_score > best_hough_score:
                            best_hough_score = hough_score
                            best_hough = (cx_h, cy_h, r_h, bright_pixels, mean_l_inside)

                if best_hough is not None:
                    cx_h, cy_h, r_h, bright_pixels, mean_l_inside = best_hough
                    
                    final_hough_mask = np.zeros_like(mask)
                    cv2.circle(final_hough_mask, (cx_h, cy_h), r_h, 255, -1)
                    mean_bgr = cv2.mean(frame, mask=final_hough_mask)[:3]

                    best = {
                        "center": (cx_h + offset_x, cy_h + offset_y),
                        "radius_est": r_h,
                        "pixel_count": bright_pixels,
                        "mean_bgr": mean_bgr,
                        "mean_value": mean_l_inside,
                        "local_contrast": 0.0,
                        "circularity": 1.0,
                        "solidity": 1.0,
                        "score": round(best_hough_score, 2),
                        "method": "hough"
                    }

    return best, l_channel, mask, best_contour, mask

# ==========================================================
# Detection Parameters for Dark Clouds
# ==========================================================
MIN_SUN_AREA_RATIO = 0.0002
MAX_SUN_AREA_RATIO = 0.6
SUN_BRIGHTNESS_RATIO = 0.98


def detect_sun_dark(frame, offset_x=0, offset_y=0):
    if frame is None or frame.size == 0:
        return None, None, None, None, None

    # ۱. بررسی اولیه وضعیت شب/روز
    is_day, details = analyze_sky_and_clouds(frame)
    
    if not is_day:
        return None, None, None, None, None

    h, w = frame.shape[:2]

    # محدود کردن ناحیه پردازش به ۹۰٪ بالای تصویر
    if offset_x == 0 and offset_y == 0:
        sky_limit = int(h * 0.90)
        frame = frame[:sky_limit]
        h, w = frame.shape[:2]

    total_pixels = h * w

    # ==========================================================
    # پردازش مخصوص حالت: ابر سیاه / تاریک (Dark Clouds)
    # ==========================================================
    if details["cloud_status"] == "Dark Clouds":
        # تبدیل به فضای رنگی LAB و استخراج کانال L
        lab = cv2.cvtColor(frame, cv2.COLOR_BGR2LAB)
        l_channel = lab[:, :, 0]
        
        blur_l = cv2.GaussianBlur(l_channel, (9, 9), 0)

        # آستانه‌گذاری پویا
        max_val = np.max(blur_l)
        custom_thresh_val = int(max_val * SUN_BRIGHTNESS_RATIO)
        _, mask_l = cv2.threshold(l_channel, custom_thresh_val, 255, cv2.THRESH_BINARY)

        # عملیات مورفولوژی ساده جهت پاک‌سازی نویزها
        kernel1 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
        kernel2 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (11, 11))

        mask = cv2.morphologyEx(mask_l, cv2.MORPH_OPEN, kernel1)
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel2)

        # ورودی برای هاف
        hough_input = cv2.GaussianBlur(mask, (5, 5), 0)

        min_r = int(np.sqrt(total_pixels * MIN_SUN_AREA_RATIO / np.pi))
        max_r = int(np.sqrt(total_pixels * MAX_SUN_AREA_RATIO / np.pi))

        # اجرای Hough Circles
        circles = cv2.HoughCircles(
            hough_input,
            cv2.HOUGH_GRADIENT,
            dp=1.2,
            minDist=30,
            param1=80,
            param2=10,
            minRadius=max(5, min_r),
            maxRadius=max_r
        )

        best = None
        best_score = -1.0

        if circles is not None:
            circles = np.uint16(np.around(circles))

            for c in circles[0, :]:
                cx_h, cy_h, r_h = int(c[0]), int(c[1]), int(c[2])

                if 0 <= cx_h < w and 0 <= cy_h < h and r_h > 0:
                    temp_circle_mask = np.zeros_like(mask)
                    cv2.circle(temp_circle_mask, (cx_h, cy_h), r_h, 255, -1)

                    # محاسبه پیکسل‌های روشن داخل دایره
                    overlap = cv2.bitwise_and(mask, temp_circle_mask)
                    bright_pixels = cv2.countNonZero(overlap)

                    if bright_pixels == 0:
                        continue

                    # ۱. تراکم پیکسل‌های روشن درون دایره (Density)
                    circle_area = np.pi * (r_h ** 2)
                    density = bright_pixels / circle_area

                    # ۲. روشنایی میانگین داخل دایره
                    mean_l_inside = cv2.mean(l_channel, mask=temp_circle_mask)[0]

                    # ۳. محاسبه کنتراست محلی (اختلاف روشنایی دایره با نوار اطراف آن)
                    ring_mask = cv2.dilate(
                        temp_circle_mask,
                        cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (21, 21))
                    )
                    ring_mask = cv2.subtract(ring_mask, temp_circle_mask)
                    bg_l_value = cv2.mean(l_channel, mask=ring_mask)[0]
                    local_contrast = mean_l_inside - bg_l_value

                    # منطق امتیازدهی کد اولیه
                    score = (bright_pixels * 0.4) + (density * 300.0) + (local_contrast * 5.0)

                    if score > best_score:
                        best_score = score
                        
                        final_mask = np.zeros_like(mask)
                        cv2.circle(final_mask, (cx_h, cy_h), r_h, 255, -1)
                        mean_bgr = cv2.mean(frame, mask=final_mask)[:3]

                        best = {
                            "center": (cx_h + offset_x, cy_h + offset_y),
                            "radius_est": r_h,
                            "pixel_count": bright_pixels,
                            "mean_bgr": mean_bgr,
                            "mean_value": mean_l_inside,
                            "local_contrast": local_contrast,
                            "circularity": 1.0,
                            "solidity": 1.0,
                            "score": round(score, 2),
                            "method": "hough_dark"
                        }

        return best, l_channel, mask, None, mask
   